# Week 5-1 — Hybrid Search (BM25 + Dense + RRF)

**목적**: Dense(e5) 단독 검색의 약점 — 로우레벨 검증에서 확인한 "고유명사·정확 용어(HER2, BRCA 등) 변별력 부족" — 을 키워드 매칭(BM25)으로 보완한다. BM25 / Dense / Hybrid **3가지 retriever를 같은 질문 세트로 격리 비교**한다.

- **Baseline**: 5-0에서 채택한 **P1(참고문헌 제거) 인덱스**. BM25는 키워드 매칭이라 참고문헌의 저자명·저널명에 특히 오염되기 쉬우므로, 제거를 먼저 완료한 순서가 유효하다.
- **RRF 직접 구현**: 결합 로직을 명시적으로 보이기 위해 EnsembleRetriever 대신 Reciprocal Rank Fusion을 직접 구현 (k=60 표준).
- **한국어 토큰화**: BM25는 공백 분리로는 한국어에 무력하므로 kiwipiepy 형태소 분석 적용.
- judge: gpt-4o-mini (5주차 통일). 이 실험 안에서만 상대 비교.

---
## 1. 설정 + 5-0 전처리 상수 (재현성 위해 하드코딩)

In [1]:
from pathlib import Path
import os, json, re
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_EVAL = PROJECT_ROOT / "data" / "eval"
VECTOR_ROOT = PROJECT_ROOT / "data" / "vector_store"

load_dotenv(PROJECT_ROOT / ".env"); load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 없음"

EMBEDDING_MODEL = "intfloat/multilingual-e5-base"
GEN_MODEL = "gpt-4o-mini"
JUDGE_MODEL = "gpt-4o-mini"
TOP_K = 5
RRF_K = 60  # RRF 표준 상수

# 청킹: 4주차 확정 G2
CHUNK_SIZE_BY_LANG    = {"ko": 540, "en": 620, "unknown": 580}
CHUNK_OVERLAP_BY_LANG = {"ko": 80,  "en": 90,  "unknown": 85}

# ---- 5-0에서 확정한 전처리 (P1) — 재현성 위해 최종값 하드코딩 ----
CUT_FROM_PAGE = {
    "esmo_breast_cancer_patient_guide_korean.pdf": 60,
    "ncc_breast_cancer_screening_guideline_2015.pdf": 109,
    "nccn_metastatic_breast_cancer_patient.pdf": 63,
}
# kbcs: 인용-밀도 필터(threshold 0.35)로 확정한 82개 페이지
KBCS_REF_PAGES = set([
    109, 111, 115, 117, 121, 123, 124, 125, 126, 127, 129, 130, 131, 132, 135,
    137, 139, 141, 172, 173, 174, 223, 232, 233, 234, 235, 236, 237, 238, 240,
    241, 242, 243, 244, 245, 247, 248, 249, 250, 251, 252,
    50, 51, 53, 56, 100, 101, 103, 104, 108, 110, 112, 113, 114, 116, 118,
    119, 120, 122, 128, 133, 134, 136, 138, 140, 163, 164, 167, 168, 169, 170,
    171, 221, 222, 224, 227, 228, 229, 230, 231, 239, 246,
])
KBCS_NAME = "kbcs_korean_breast_cancer_guideline_2023.pdf"
print(f"전처리 상수: 컷 {len(CUT_FROM_PAGE)}종 / kbcs 제외 {len(KBCS_REF_PAGES)}p")

전처리 상수: 컷 3종 / kbcs 제외 82p


---
## 2. P1 chunk 재구성 (5-0과 동일 규칙)

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

manifest_path = DATA_RAW / "metadata" / "manifest.json"
meta_lookup = {}
if manifest_path.exists():
    with open(manifest_path, encoding="utf-8") as f:
        meta_lookup = {m["filename"]: m for m in json.load(f) if m.get("downloaded")}

def guess_lang(text):
    return "ko" if len(re.findall(r"[\uac00-\ud7a3]", text)) > 20 else "en"

def load_docs_p1():
    out = []
    for pdf in sorted((DATA_RAW / "pdf").rglob("*.pdf")):
        cut = CUT_FROM_PAGE.get(pdf.name)
        for d in PyMuPDFLoader(str(pdf)).load():
            pg = d.metadata.get("page", 0)
            if cut is not None and pg >= cut:
                continue
            if pdf.name == KBCS_NAME and pg in KBCS_REF_PAGES:
                continue
            extra = meta_lookup.get(pdf.name, {})
            d.metadata.update({
                "filename": pdf.name, "org": extra.get("org", pdf.parent.name),
                "title": extra.get("title", pdf.stem),
                "language": extra.get("language", guess_lang(d.page_content)),
                "page": pg,
            })
            out.append(d)
    return out

SEPARATORS = ["\n\n", "\n", ". ", " ", ""]

def split_docs(docs):
    out = []
    for lang in set(d.metadata.get("language", "unknown") for d in docs):
        size = CHUNK_SIZE_BY_LANG.get(lang, 580); ov = CHUNK_OVERLAP_BY_LANG.get(lang, 85)
        sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=ov,
                                            separators=SEPARATORS, length_function=len)
        out.extend(sp.split_documents([d for d in docs if d.metadata.get("language") == lang]))
    return out

chunks = split_docs(load_docs_p1())
print(f"P1 chunk: {len(chunks)}개  (5-0 기준 2,240개와 일치해야 함)")

P1 chunk: 2240개  (5-0 기준 2,240개와 일치해야 함)


---
## 3. Dense retriever — 5-0의 P1 컬렉션 재사용

In [3]:
import os as _os
from huggingface_hub import snapshot_download
_os.environ.pop("HF_HUB_OFFLINE", None); _os.environ.pop("TRANSFORMERS_OFFLINE", None)
_os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

try:
    model_dir = snapshot_download(repo_id=EMBEDDING_MODEL, local_files_only=True)
except Exception:
    model_dir = snapshot_download(repo_id=EMBEDDING_MODEL, local_files_only=False, max_workers=1)

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import chromadb

embeddings = HuggingFaceEmbeddings(model_name=model_dir,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 16})

vdir = VECTOR_ROOT / "week5pre_P1_ref_removed"
coll = "breast_rag_week5pre_P1_ref_removed"
client = chromadb.PersistentClient(path=str(vdir))
if coll in [c.name for c in client.list_collections()]:
    vs = Chroma(collection_name=coll, embedding_function=embeddings, persist_directory=str(vdir))
    print(f"P1 컬렉션 재사용 ({vs._collection.count()}개)")
else:
    print("P1 컬렉션 없음 -> 신규 인덱싱 (5-0 미실행 시)")
    vs = Chroma.from_documents(documents=chunks, embedding=embeddings,
                               collection_name=coll, persist_directory=str(vdir))

def dense_search(query, k=TOP_K):
    return vs.similarity_search(query, k=k)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


P1 컬렉션 재사용 (2240개)


---
## 4. BM25 retriever — kiwipiepy 한국어 형태소 토큰화

BM25는 토큰 단위 매칭이므로 한국어는 형태소 분석이 필수다 (공백 분리로는 "유방암은/유방암의/유방암을"이 전부 다른 토큰이 됨). 영어는 소문자화 + 단순 분리.

In [4]:
from rank_bm25 import BM25Okapi
from kiwipiepy import Kiwi

kiwi = Kiwi()

def tokenize(text):
    """한국어: 형태소(내용어 위주) / 영어: 소문자 단어"""
    tokens = []
    for tok in kiwi.tokenize(text):
        # 내용어(명사·동사·형용사·외국어·숫자)만 — 조사·어미 제거로 매칭 안정화
        if tok.tag[0] in ("N", "V", "S") or tok.tag in ("SL", "SN", "XR"):
            tokens.append(tok.form.lower())
    return tokens

# 코퍼스 토큰화 (수 분 소요)
from tqdm import tqdm
corpus_tokens = [tokenize(c.page_content) for c in tqdm(chunks, desc="BM25 토큰화")]
bm25 = BM25Okapi(corpus_tokens)
print(f"BM25 인덱스 완료: {len(corpus_tokens)}개 chunk")

def bm25_search(query, k=TOP_K):
    scores = bm25.get_scores(tokenize(query))
    top_idx = sorted(range(len(scores)), key=lambda i: -scores[i])[:k]
    return [chunks[i] for i in top_idx]

BM25 토큰화: 100%|██████████| 2240/2240 [00:04<00:00, 489.70it/s] 

BM25 인덱스 완료: 2240개 chunk


---
## 5. Hybrid — RRF(Reciprocal Rank Fusion) 직접 구현

각 retriever의 **순위**만 사용해 결합: `score(d) = Σ 1/(k + rank)`, k=60. 점수 스케일이 다른 두 검색(BM25 점수 vs cosine)을 정규화 없이 안전하게 합치는 표준 방식.

In [5]:
def _key(doc):
    # 동일 chunk 식별 (문서명+페이지+내용 앞부분)
    return (doc.metadata.get("filename"), doc.metadata.get("page"), doc.page_content[:80])

def hybrid_search(query, k=TOP_K, fetch_k=20):
    """Dense·BM25 각각 fetch_k개 가져와 RRF로 융합 후 상위 k개"""
    dense_docs = dense_search(query, k=fetch_k)
    bm25_docs = bm25_search(query, k=fetch_k)
    scores, registry = {}, {}
    for rank, d in enumerate(dense_docs):
        kk = _key(d); registry[kk] = d
        scores[kk] = scores.get(kk, 0) + 1.0 / (RRF_K + rank + 1)
    for rank, d in enumerate(bm25_docs):
        kk = _key(d); registry[kk] = d
        scores[kk] = scores.get(kk, 0) + 1.0 / (RRF_K + rank + 1)
    top = sorted(scores, key=lambda kk: -scores[kk])[:k]
    return [registry[kk] for kk in top]

RETRIEVERS = {
    "R1_dense": dense_search,
    "R2_bm25": bm25_search,
    "R3_hybrid": hybrid_search,
}
print("retrievers:", list(RETRIEVERS.keys()))

retrievers: ['R1_dense', 'R2_bm25', 'R3_hybrid']


---
## 6. 로우레벨 sanity check 

RAGAS 돌리기 전에, 세 retriever가 실제로 다른 걸 가져오는지 — 특히 **고유명사 질문에서 BM25가 강한지** — 눈으로 먼저 확인한다.

In [6]:
test_queries = [
    "HER2 양성 유방암은 어떤 표적 치료를 하나요?",   # 고유명사 -> BM25 강세 기대
    "유방암 1기와 2기의 차이는 무엇인가요?",          # 4주차부터 고질 실패 문항
]
for q in test_queries:
    print("=" * 70)
    print(f"Q: {q}")
    for name, fn in RETRIEVERS.items():
        docs = fn(q, 3) if name != "R3_hybrid" else fn(q, k=3)
        print(f"\n  --- {name} top-3")
        for d in docs:
            head = d.page_content.strip().replace("\n", " ")[:80]
            print(f"    [{d.metadata.get('filename','?')[:20]} p.{d.metadata.get('page','?')}] {head}")
    print()

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Q: HER2 양성 유방암은 어떤 표적 치료를 하나요?

  --- R1_dense top-3
    [esmo_breast_cancer_p p.26] 항 HER2 요법   HER2 양성 유방암은 일반적으로 정맥 주입 또는 피하 주사와 화학 요법을 통해 항 HER2 제제인  트라스투주맙으로 치료
    [esmo_breast_cancer_p p.18] 성장을 촉진하기 위해 추가되는 많은 수의 수용체 (ER 또는 PgR) 를 가지고 있습니다. ER 이 발현되는  종양을 ER 양성 종양이라고 하며
    [esmo_breast_cancer_p p.7] 중요합니다.  HER2 는 세포의 성장에 관여하는 또다른 수용체이며, 유방암의 약 20% 에 존재합니다. HER2  발현이가 높은 종양은 항 H

  --- R2_bm25 top-3
    [kbcs_korean_breast_c p.72] 2023 The 10 th Korean Clinical Practice Guideline for Breast Cancer | 73 제2장 조기 
    [kbcs_korean_breast_c p.149] p=0.0063). 하지만 3제 병용요법의 우수한 효과만큼 부작용도 심하였고, 특히 3등급 이상의 설사가 약 13%의 환 자에서 보고되었다(70
    [kbcs_korean_breast_c p.148] 킨다는 증거는 있지만 충분하지 않고, 전체 생존기간의 차이는 미약하기 때문에 항암화학요법 의 기간은 부작용과 전반적인 삶의 질에 미치는 영향을 

  --- R3_hybrid top-3
    [kbcs_korean_breast_c p.72] 2023 The 10 th Korean Clinical Practice Guideline for Breast Cancer | 73 제2장 조기 
    [kbcs_korean_breast_c p.149] p=0.0063). 하지만 3제 병용요법의 우수한 효과만큼 부작용도 심하였고, 특히 3등급 이상의 설사가 약 13%의 환 자에서 보고되었다(

### sanity 관찰 메모 (직접 채우기)
- 고유명사 질문에서 BM25가 Dense와 다른(더 정확한) chunk를 가져오는가?
- Hybrid가 양쪽의 결과를 실제로 섞는가?

---
## 7. RAGAS 비교 — Dense vs BM25 vs Hybrid (judge=gpt-4o-mini)

In [7]:
import pandas as pd
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model=GEN_MODEL, temperature=0)
RAG_PROMPT = ChatPromptTemplate.from_template(
    """당신은 유방암 정보 검색 보조 시스템입니다.

아래 [참고 문서]만 사용해서 [질문]에 답변하세요. 문서에 없는 내용은 추측하지 말고 "제공된 문서에서 확인할 수 없습니다"라고 답하세요.
답변 마지막에는 반드시 다음 두 가지를 포함하세요:
1. 출처: 참고한 문서명과 페이지 (예: 출처: 국립암센터 유방암 검진 권고안, p.5)
2. 면책 문구: "이 답변은 일반 정보 제공 목적이며, 실제 진단·치료는 반드시 의료진과 상의하세요."

[참고 문서]
{context}

[질문]
{question}

[답변]""")

def format_context(docs):
    return "\n\n---\n\n".join(
        f"[{i}] 출처: {d.metadata.get('org','?')} / {d.metadata.get('title','?')} / p.{d.metadata.get('page','?')}\n{d.page_content}"
        for i, d in enumerate(docs, 1))

golden = pd.read_csv(DATA_EVAL / "golden_set_v1.csv").to_dict("records")
print(f"golden_set: {len(golden)}문항")

golden_set: 30문항


In [8]:
import nest_asyncio; nest_asyncio.apply()
from ragas import evaluate, EvaluationDataset
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

ragas_llm = LangchainLLMWrapper(ChatOpenAI(model=JUDGE_MODEL, temperature=0))
ragas_emb = LangchainEmbeddingsWrapper(embeddings)
METRICS = [faithfulness, answer_relevancy, context_precision]
METRIC_COLS = ["faithfulness", "answer_relevancy", "context_precision"]

def run_retriever(name, fn):
    rows = []
    for g in tqdm(golden, desc=f"RAG[{name}]"):
        docs = fn(g["question"])
        ans = (llm | StrOutputParser()).invoke(
            RAG_PROMPT.format(context=format_context(docs), question=g["question"]))
        rows.append({"user_input": g["question"], "response": ans,
                     "retrieved_contexts": [d.page_content for d in docs],
                     "reference": g.get("ground_truth", "")})
    ds = EvaluationDataset.from_list(rows)
    df = evaluate(dataset=ds, metrics=METRICS, llm=ragas_llm, embeddings=ragas_emb).to_pandas()
    df.to_csv(DATA_PROCESSED / f"week5_ragas_{name}.csv", index=False, encoding="utf-8-sig")
    return df

score_tables = {}
for name, fn in RETRIEVERS.items():
    score_tables[name] = run_retriever(name, fn)
    print(f"{name}: 완료")

RAG[R1_dense]:   0%|          | 0/30 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
RAG[R1_dense]: 100%|██████████| 30/30 [01:38<00:00,  3.28s/it]


Evaluating:   0%|          | 0/90 [00:00<?, ?it/s]

R1_dense: 완료


RAG[R2_bm25]: 100%|██████████| 30/30 [01:21<00:00,  2.71s/it]


Evaluating:   0%|          | 0/90 [00:00<?, ?it/s]

R2_bm25: 완료


RAG[R3_hybrid]: 100%|██████████| 30/30 [01:27<00:00,  2.93s/it]


Evaluating:   0%|          | 0/90 [00:00<?, ?it/s]

R3_hybrid: 완료


---
## 8. 비교 — 전체·언어별·문항별

In [9]:
def _lang(q): return "KO" if re.search("[가-힣]", str(q)) else "EN"

rows = []
for name in RETRIEVERS:
    df = score_tables[name]
    row = {"retriever": name}
    for c in METRIC_COLS:
        row[c] = round(df[c].mean(), 4)
    rows.append(row)
cmp = pd.DataFrame(rows)
for c in METRIC_COLS:
    cmp[c + "_delta"] = (cmp[c] - cmp[c].iloc[0]).round(4)  # Dense 기준 델타
cmp.to_csv(DATA_PROCESSED / "week5_hybrid_comparison.csv", index=False, encoding="utf-8-sig")
print(cmp.to_string())

print("\n--- 언어별 context_precision ---")
for L in ["KO", "EN"]:
    for name in RETRIEVERS:
        df = score_tables[name].copy(); df["_l"] = df["user_input"].apply(_lang)
        sub = df[df["_l"] == L]
        print(f"  [{L}] {name:10s} CP={sub['context_precision'].mean():.4f}")
    print()

   retriever  faithfulness  answer_relevancy  context_precision  faithfulness_delta  answer_relevancy_delta  context_precision_delta
0   R1_dense        0.7487            0.6120             0.9072              0.0000                  0.0000                   0.0000
1    R2_bm25        0.6542            0.4482             0.8579             -0.0945                 -0.1638                  -0.0493
2  R3_hybrid        0.7784            0.6090             0.9378              0.0297                 -0.0030                   0.0306

--- 언어별 context_precision ---
  [KO] R1_dense   CP=0.9602
  [KO] R2_bm25    CP=0.9040
  [KO] R3_hybrid  CP=0.9883

  [EN] R1_dense   CP=0.8011
  [EN] R2_bm25    CP=0.7656
  [EN] R3_hybrid  CP=0.8367



In [10]:
# 문항별: Hybrid가 Dense 대비 개선/악화된 문항
d_ = score_tables["R1_dense"][["user_input", "context_precision"]]
h_ = score_tables["R3_hybrid"][["user_input", "context_precision"]]
m = d_.merge(h_, on="user_input", suffixes=("_dense", "_hybrid"))
m["delta"] = (m["context_precision_hybrid"] - m["context_precision_dense"]).round(3)
m["lang"] = m["user_input"].apply(_lang)

print("=== Hybrid로 CP가 크게 변한 문항 (|delta| > 0.1) ===")
for _, r in m[abs(m["delta"]) > 0.1].sort_values("delta", ascending=False).iterrows():
    print(f"  {r['delta']:+.2f} ({r['lang']}) {str(r['user_input'])[:44]}")
print(f"\n개선 {(m['delta']>0.05).sum()} / 악화 {(m['delta']<-0.05).sum()} / 동일 {(abs(m['delta'])<=0.05).sum()}")

# 고질 실패 문항 추적
print("\n=== 고질 문항 확인 ===")
for kw in ["1기와 2기", "식이요법", "감시림프절"]:
    row = m[m["user_input"].str.contains(kw, na=False)]
    if len(row):
        r = row.iloc[0]
        print(f"  {kw}: dense={r['context_precision_dense']:.2f} -> hybrid={r['context_precision_hybrid']:.2f}")

=== Hybrid로 CP가 크게 변한 문항 (|delta| > 0.1) ===
  +0.50 (KO) 유방암 1기와 2기의 차이는 무엇인가요?
  +0.42 (EN) What are the different types of breast biops
  +0.14 (EN) What do the BI-RADS assessment categories 0 
  +0.13 (KO) 타목시펜의 부작용에는 어떤 것이 있나요?
  -0.25 (EN) What factors put a person at increased risk 

개선 5 / 악화 2 / 동일 23

=== 고질 문항 확인 ===
  1기와 2기: dense=0.45 -> hybrid=0.95
  식이요법: dense=0.89 -> hybrid=0.92
  감시림프절: dense=1.00 -> hybrid=1.00


---
## 판정 & 기록

- **BM25 단독 vs Dense 단독**: Dense 우세 (CP 0.907 vs 0.858, 문항 단위 6승 3패). 예상과 달리 고유명사(HER2) 질문은 Dense가 이미 잘 처리했고, BM25의 진짜 기여는 "1기/2기" 같은 **짧고 정확한 표현 매칭**이었다. 다만 BM25는 단독으로는 취약 (식이요법 CP 0.00, answer_relevancy 0.448) — 융합 재료로서 가치.
- **Hybrid 효과**: Dense 대비 CP +0.031 (0.907→0.938), 두 언어 모두 최고 (KO 0.988 / EN 0.837). 문항 단위 개선 5 / 악화 2 / 동일 23. 핵심은 **두 검색기의 실패 패턴이 달라 RRF 융합이 서로의 구멍을 상쇄**한 것 — 식이요법(bm25 0.00 → hybrid 0.92), 1기·2기(dense 0.45 → hybrid 0.95).
- **고질 문항(1기·2기)**: **해소.** 4주차 모든 청킹 전략에서 실패하던 문항이 dense 0.45 → hybrid 0.95. 청킹으로 못 풀던 문제가 retrieval 방식으로 풀림. 잔여 관찰: 악화 2문항은 모두 영어(increased risk −0.25 등, RRF의 영어 순위 흔들림) — Error Case 분석 후보로 이월.
- **판정**: **Hybrid 채택.** 5-2(Rerank)의 1차 검색은 **Hybrid top-20** → reranker로 top-5 재정렬.
- decision_log(Week 5)에 요약 추가: "Retrieval = Hybrid(BM25+Dense, RRF k=60) 채택 — CP 0.907→0.938, 고질 문항(1기·2기) 0.45→0.95 해소. 두 검색기의 실패 패턴이 달라 융합이 상쇄."